In [50]:
# assignment_pyspark.py
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

spark = SparkSession.builder.master("local[*]").appName("PySpark Assignment").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# --- paste dataset creation code here ---
# --- Employees ---
employees_data = [
    (1,  "Alice",   "Engineering", "India",  95000, "2019-03-15", True),
    (2,  "Bob",     "Marketing",   "USA",    72000, "2020-07-01", True),
    (3,  "Carol",   "Engineering", "USA",    110000,"2018-11-20", True),
    (4,  "David",   "HR",          "India",  60000, "2021-01-10", False),
    (5,  "Eva",     "Engineering", "UK",     98000, "2017-06-05", True),
    (6,  "Frank",   "Marketing",   "India",  68000, "2022-03-22", True),
    (7,  "Grace",   "HR",          "USA",    63000, "2020-09-14", True),
    (8,  "Hank",    "Engineering", "UK",     120000,"2016-12-01", False),
    (9,  "Ivy",     "Marketing",   "UK",     75000, "2021-05-30", True),
    (10, "Jake",    "HR",          "USA",    58000, "2023-02-18", True),
]
employees_schema = StructType([
    StructField("emp_id",     IntegerType(), False),
    StructField("name",       StringType(),  False),
    StructField("department", StringType(),  False),
    StructField("country",    StringType(),  False),
    StructField("salary",     IntegerType(), False),
    StructField("join_date",  StringType(),  False),
    StructField("is_active",  BooleanType(), False),
])
employees = spark.createDataFrame(employees_data, employees_schema)

# --- Orders ---
orders_data = [
    (101, 1, "2024-01-05", 500.0,  "Electronics"),
    (102, 3, "2024-01-18", 1200.0, "Electronics"),
    (103, 2, "2024-02-10", 300.0,  "Clothing"),
    (104, 1, "2024-02-20", 850.0,  "Electronics"),
    (105, 5, "2024-03-01", 2000.0, "Furniture"),
    (106, 3, "2024-03-15", 450.0,  "Clothing"),
    (107, 7, "2024-03-22", 150.0,  "Clothing"),
    (108, 2, "2024-04-10", 700.0,  "Electronics"),
    (109, 9, "2024-04-25", 950.0,  "Furniture"),
    (110, 1, "2024-05-02", 1100.0, "Electronics"),
    (111, 4, "2024-05-15", 200.0,  "Clothing"),
    (112, 6, "2024-06-01", 600.0,  "Electronics"),
]
orders_schema = StructType([
    StructField("order_id",   IntegerType(), False),
    StructField("emp_id",     IntegerType(), False),
    StructField("order_date", StringType(),  False),
    StructField("amount",     DoubleType(),  False),
    StructField("category",   StringType(),  False),
])
orders = spark.createDataFrame(orders_data, orders_schema)



In [51]:
employees.show()
orders.show()


+------+-----+-----------+-------+------+----------+---------+
|emp_id| name| department|country|salary| join_date|is_active|
+------+-----+-----------+-------+------+----------+---------+
|     1|Alice|Engineering|  India| 95000|2019-03-15|     true|
|     2|  Bob|  Marketing|    USA| 72000|2020-07-01|     true|
|     3|Carol|Engineering|    USA|110000|2018-11-20|     true|
|     4|David|         HR|  India| 60000|2021-01-10|    false|
|     5|  Eva|Engineering|     UK| 98000|2017-06-05|     true|
|     6|Frank|  Marketing|  India| 68000|2022-03-22|     true|
|     7|Grace|         HR|    USA| 63000|2020-09-14|     true|
|     8| Hank|Engineering|     UK|120000|2016-12-01|    false|
|     9|  Ivy|  Marketing|     UK| 75000|2021-05-30|     true|
|    10| Jake|         HR|    USA| 58000|2023-02-18|     true|
+------+-----+-----------+-------+------+----------+---------+

+--------+------+----------+------+-----------+
|order_id|emp_id|order_date|amount|   category|
+--------+------+----

# Part 1

In [52]:
# 1.1 Print the schema of employees and orders using printSchema().

employees.printSchema()
orders.printSchema()



root
 |-- emp_id: integer (nullable = false)
 |-- name: string (nullable = false)
 |-- department: string (nullable = false)
 |-- country: string (nullable = false)
 |-- salary: integer (nullable = false)
 |-- join_date: string (nullable = false)
 |-- is_active: boolean (nullable = false)

root
 |-- order_id: integer (nullable = false)
 |-- emp_id: integer (nullable = false)
 |-- order_date: string (nullable = false)
 |-- amount: double (nullable = false)
 |-- category: string (nullable = false)



In [53]:
# 1.2 Cast join_date (currently a string) in employees to DateType. Store the result back as employees.

employees=employees.withColumn("join_date", col("join_date").cast("date"))
employees.printSchema()

# alternate method

employees.withColumn("join_date", to_date(col("join_date"))).printSchema()

root
 |-- emp_id: integer (nullable = false)
 |-- name: string (nullable = false)
 |-- department: string (nullable = false)
 |-- country: string (nullable = false)
 |-- salary: integer (nullable = false)
 |-- join_date: date (nullable = true)
 |-- is_active: boolean (nullable = false)

root
 |-- emp_id: integer (nullable = false)
 |-- name: string (nullable = false)
 |-- department: string (nullable = false)
 |-- country: string (nullable = false)
 |-- salary: integer (nullable = false)
 |-- join_date: date (nullable = true)
 |-- is_active: boolean (nullable = false)



In [54]:

# 1.3 Cast order_date in orders to DateType. Store back as orders.

orders=orders.withColumn("order_date", col("order_date").cast("date"))
orders.printSchema()


root
 |-- order_id: integer (nullable = false)
 |-- emp_id: integer (nullable = false)
 |-- order_date: date (nullable = true)
 |-- amount: double (nullable = false)
 |-- category: string (nullable = false)



In [55]:
# 1.4 Add a column years_of_service to employees that computes the number of full years between join_date and today (current_date()).

employees=employees.withColumn("years_of_service",floor(datediff(current_date(),col("join_date"))/365))
employees.show()

+------+-----+-----------+-------+------+----------+---------+----------------+
|emp_id| name| department|country|salary| join_date|is_active|years_of_service|
+------+-----+-----------+-------+------+----------+---------+----------------+
|     1|Alice|Engineering|  India| 95000|2019-03-15|     true|               7|
|     2|  Bob|  Marketing|    USA| 72000|2020-07-01|     true|               5|
|     3|Carol|Engineering|    USA|110000|2018-11-20|     true|               7|
|     4|David|         HR|  India| 60000|2021-01-10|    false|               5|
|     5|  Eva|Engineering|     UK| 98000|2017-06-05|     true|               8|
|     6|Frank|  Marketing|  India| 68000|2022-03-22|     true|               4|
|     7|Grace|         HR|    USA| 63000|2020-09-14|     true|               5|
|     8| Hank|Engineering|     UK|120000|2016-12-01|    false|               9|
|     9|  Ivy|  Marketing|     UK| 75000|2021-05-30|     true|               4|
|    10| Jake|         HR|    USA| 58000

In [56]:

# 1.5 Add a boolean column is_senior that is True when years_of_service >= 5.

employees= employees.withColumn("is_senior",col("years_of_service")>=5)
employees.show()

+------+-----+-----------+-------+------+----------+---------+----------------+---------+
|emp_id| name| department|country|salary| join_date|is_active|years_of_service|is_senior|
+------+-----+-----------+-------+------+----------+---------+----------------+---------+
|     1|Alice|Engineering|  India| 95000|2019-03-15|     true|               7|     true|
|     2|  Bob|  Marketing|    USA| 72000|2020-07-01|     true|               5|     true|
|     3|Carol|Engineering|    USA|110000|2018-11-20|     true|               7|     true|
|     4|David|         HR|  India| 60000|2021-01-10|    false|               5|     true|
|     5|  Eva|Engineering|     UK| 98000|2017-06-05|     true|               8|     true|
|     6|Frank|  Marketing|  India| 68000|2022-03-22|     true|               4|    false|
|     7|Grace|         HR|    USA| 63000|2020-09-14|     true|               5|     true|
|     8| Hank|Engineering|     UK|120000|2016-12-01|    false|               9|     true|
|     9|  

# Part 2

In [57]:
# 2.1 Select all active employees (is_active = True) in the Engineering department. Show only name, salary, and country.

employees.select("name","salary","country")\
    .where((col("department")=="Engineering") & (col("is_active")=="true")).show()


+-----+------+-------+
| name|salary|country|
+-----+------+-------+
|Alice| 95000|  India|
|Carol|110000|    USA|
|  Eva| 98000|     UK|
+-----+------+-------+



In [58]:
# 2.2 Find all employees with a salary between 70,000 and 100,000 (inclusive). Sort by salary descending.

employees.filter((col("salary").between(70000,100000)))\
    .sort(col("salary").desc()).show()

+------+-----+-----------+-------+------+----------+---------+----------------+---------+
|emp_id| name| department|country|salary| join_date|is_active|years_of_service|is_senior|
+------+-----+-----------+-------+------+----------+---------+----------------+---------+
|     5|  Eva|Engineering|     UK| 98000|2017-06-05|     true|               8|     true|
|     1|Alice|Engineering|  India| 95000|2019-03-15|     true|               7|     true|
|     9|  Ivy|  Marketing|     UK| 75000|2021-05-30|     true|               4|    false|
|     2|  Bob|  Marketing|    USA| 72000|2020-07-01|     true|               5|     true|
+------+-----+-----------+-------+------+----------+---------+----------------+---------+



In [59]:
# 2.3 Add a column salary_band:

# "Low" if salary < 65,000
# "Mid" if 65,000 ≤ salary < 100,000
# "High" if salary ≥ 100,000
# Use when / otherwise.

employees=employees.withColumn("salary_band",when(col("salary")<65000,"Low")
                    .when((col("salary")>=65000) & (col("salary")<100000),"Mid")
                     .otherwise("High"))
employees.show()

+------+-----+-----------+-------+------+----------+---------+----------------+---------+-----------+
|emp_id| name| department|country|salary| join_date|is_active|years_of_service|is_senior|salary_band|
+------+-----+-----------+-------+------+----------+---------+----------------+---------+-----------+
|     1|Alice|Engineering|  India| 95000|2019-03-15|     true|               7|     true|        Mid|
|     2|  Bob|  Marketing|    USA| 72000|2020-07-01|     true|               5|     true|        Mid|
|     3|Carol|Engineering|    USA|110000|2018-11-20|     true|               7|     true|       High|
|     4|David|         HR|  India| 60000|2021-01-10|    false|               5|     true|        Low|
|     5|  Eva|Engineering|     UK| 98000|2017-06-05|     true|               8|     true|        Mid|
|     6|Frank|  Marketing|  India| 68000|2022-03-22|     true|               4|    false|        Mid|
|     7|Grace|         HR|    USA| 63000|2020-09-14|     true|               5|   

In [60]:
# 2.4 Rename name to employee_name and drop the is_active column. Show the resulting schema.

employees.withColumnRenamed("name","employee_name").drop(col("is_active"))
employees=employees.withColumnRenamed("name","employee_name")


# part 3

In [61]:
# 3.1 Calculate the average salary per department. Round to 2 decimal places.

employees.groupBy("department")\
    .agg(round(avg("salary"),2).alias("avg_salary")).show()


+-----------+----------+
| department|avg_salary|
+-----------+----------+
|Engineering|  105750.0|
|  Marketing|  71666.67|
|         HR|  60333.33|
+-----------+----------+



In [62]:

# 3.2 For each department, find:

# Total headcount
# Number of active employees
# Minimum and maximum salary

employees.groupBy("department")\
    .agg(count("emp_id"),count(when(col("is_active")=="true",col("emp_id"))),max("salary"),min("salary"))\
    .show()


+-----------+-------------+---------------------------------------------------+-----------+-----------+
| department|count(emp_id)|count(CASE WHEN (is_active = true) THEN emp_id END)|max(salary)|min(salary)|
+-----------+-------------+---------------------------------------------------+-----------+-----------+
|Engineering|            4|                                                  3|     120000|      95000|
|  Marketing|            3|                                                  3|      75000|      68000|
|         HR|            3|                                                  2|      63000|      58000|
+-----------+-------------+---------------------------------------------------+-----------+-----------+



In [63]:
# 3.3 Find departments where the average salary is above 80,000.

employees.groupBy("department")\
    .agg(avg("salary").alias("avg_salary")).filter(col("avg_salary")>80000).show()


+-----------+----------+
| department|avg_salary|
+-----------+----------+
|Engineering|  105750.0|
+-----------+----------+



In [64]:
# 3.4 From orders, compute total revenue and average order value per category. Sort by total revenue descending.

orders.groupBy("category")\
        .agg(sum("amount").alias("total_revenue"),avg("amount").alias("avg_order_value")).sort(col("total_revenue").desc()).show()

+-----------+-------------+---------------+
|   category|total_revenue|avg_order_value|
+-----------+-------------+---------------+
|Electronics|       4950.0|          825.0|
|  Furniture|       2950.0|         1475.0|
|   Clothing|       1100.0|          275.0|
+-----------+-------------+---------------+



# part 4

In [65]:
# 4.1 Perform an inner join of orders with employees on emp_id. Select: order_id, employee_name (renamed from name), department, amount, category, order_date.
emp_df=employees.join(orders,employees["emp_id"]==orders["emp_id"],"inner")\
    .select("order_id", "employee_name", "department", "amount", "category", "order_date")

emp_df.show()



+--------+-------------+-----------+------+-----------+----------+
|order_id|employee_name| department|amount|   category|order_date|
+--------+-------------+-----------+------+-----------+----------+
|     101|        Alice|Engineering| 500.0|Electronics|2024-01-05|
|     104|        Alice|Engineering| 850.0|Electronics|2024-02-20|
|     110|        Alice|Engineering|1100.0|Electronics|2024-05-02|
|     103|          Bob|  Marketing| 300.0|   Clothing|2024-02-10|
|     108|          Bob|  Marketing| 700.0|Electronics|2024-04-10|
|     102|        Carol|Engineering|1200.0|Electronics|2024-01-18|
|     106|        Carol|Engineering| 450.0|   Clothing|2024-03-15|
|     111|        David|         HR| 200.0|   Clothing|2024-05-15|
|     105|          Eva|Engineering|2000.0|  Furniture|2024-03-01|
|     112|        Frank|  Marketing| 600.0|Electronics|2024-06-01|
|     107|        Grace|         HR| 150.0|   Clothing|2024-03-22|
|     109|          Ivy|  Marketing| 950.0|  Furniture|2024-04

In [66]:
# 4.2 Find employees who have placed no orders (use a left anti join or left join + filter on null).

employees.join(orders,employees["emp_id"]==orders["emp_id"],"left_anti")\
    .show()

+------+-------------+-----------+-------+------+----------+---------+----------------+---------+-----------+
|emp_id|employee_name| department|country|salary| join_date|is_active|years_of_service|is_senior|salary_band|
+------+-------------+-----------+-------+------+----------+---------+----------------+---------+-----------+
|     8|         Hank|Engineering|     UK|120000|2016-12-01|    false|               9|     true|       High|
|    10|         Jake|         HR|    USA| 58000|2023-02-18|     true|               3|    false|        Low|
+------+-------------+-----------+-------+------+----------+---------+----------------+---------+-----------+



In [67]:
# 4.3 Find the top spender (employee with the highest total order amount). Show their name, department, and total amount.

employees.join(orders,employees["emp_id"]==orders["emp_id"],"inner")\
    .groupBy("employee_name","department")\
    .agg(sum("amount").alias("total_amount"))\
    .sort(col("total_amount").desc())\
    .select("employee_name", "department", "total_amount")\
    .show(1)

[Stage 166:===================================>                     (5 + 3) / 8]

+-------------+-----------+------------+
|employee_name| department|total_amount|
+-------------+-----------+------------+
|        Alice|Engineering|      2450.0|
+-------------+-----------+------------+
only showing top 1 row


# Part 5 — Window Functions

In [68]:
# 5.1 Rank employees within each department by salary (highest = rank 1). Use rank(). Show name, department, salary, and rank.

window= Window.partitionBy("department").orderBy(col("salary").desc())

employees.withColumn("rank",rank().over(window))\
    .select("employee_name", "department", "salary", "rank")\
    .show(truncate=False)

+-------------+-----------+------+----+
|employee_name|department |salary|rank|
+-------------+-----------+------+----+
|Hank         |Engineering|120000|1   |
|Carol        |Engineering|110000|2   |
|Eva          |Engineering|98000 |3   |
|Alice        |Engineering|95000 |4   |
|Grace        |HR         |63000 |1   |
|David        |HR         |60000 |2   |
|Jake         |HR         |58000 |3   |
|Ivy          |Marketing  |75000 |1   |
|Bob          |Marketing  |72000 |2   |
|Frank        |Marketing  |68000 |3   |
+-------------+-----------+------+----+



In [69]:
# 5.2 For each employee, compute the cumulative salary sum within their department, ordered by join_date ascending. Use sum() as a window function.

window= Window.partitionBy("department").orderBy("join_date").rowsBetween(Window.unboundedPreceding, Window.currentRow)

employees.withColumn("Cumulative_salary_sum",sum("salary").over(window))\
    .show(truncate=False)

+------+-------------+-----------+-------+------+----------+---------+----------------+---------+-----------+---------------------+
|emp_id|employee_name|department |country|salary|join_date |is_active|years_of_service|is_senior|salary_band|Cumulative_salary_sum|
+------+-------------+-----------+-------+------+----------+---------+----------------+---------+-----------+---------------------+
|8     |Hank         |Engineering|UK     |120000|2016-12-01|false    |9               |true     |High       |120000               |
|5     |Eva          |Engineering|UK     |98000 |2017-06-05|true     |8               |true     |Mid        |218000               |
|3     |Carol        |Engineering|USA    |110000|2018-11-20|true     |7               |true     |High       |328000               |
|1     |Alice        |Engineering|India  |95000 |2019-03-15|true     |7               |true     |Mid        |423000               |
|7     |Grace        |HR         |USA    |63000 |2020-09-14|true     |5     

In [70]:
# 5.3 For each order, add a column running_total — the running total of amount per emp_id ordered by order_date.

window= Window.partitionBy("emp_id").orderBy("order_date").rowsBetween(Window.unboundedPreceding, Window.currentRow)

orders.withColumn("running _total",sum("amount").over(window))\
    .show(truncate=False)

+--------+------+----------+------+-----------+--------------+
|order_id|emp_id|order_date|amount|category   |running _total|
+--------+------+----------+------+-----------+--------------+
|101     |1     |2024-01-05|500.0 |Electronics|500.0         |
|104     |1     |2024-02-20|850.0 |Electronics|1350.0        |
|110     |1     |2024-05-02|1100.0|Electronics|2450.0        |
|103     |2     |2024-02-10|300.0 |Clothing   |300.0         |
|108     |2     |2024-04-10|700.0 |Electronics|1000.0        |
|102     |3     |2024-01-18|1200.0|Electronics|1200.0        |
|106     |3     |2024-03-15|450.0 |Clothing   |1650.0        |
|111     |4     |2024-05-15|200.0 |Clothing   |200.0         |
|105     |5     |2024-03-01|2000.0|Furniture  |2000.0        |
|112     |6     |2024-06-01|600.0 |Electronics|600.0         |
|107     |7     |2024-03-22|150.0 |Clothing   |150.0         |
|109     |9     |2024-04-25|950.0 |Furniture  |950.0         |
+--------+------+----------+------+-----------+--------

In [71]:
# 5.4 Add a column prev_order_amount to orders showing the previous order amount for the same employee (use lag()). Fill nulls with 0.0.

window3=Window.partitionBy("emp_id").orderBy("order_date")

orders.withColumn("prev_order_amount",lag(col("amount"),1,0.0).over(window3)).show()


+--------+------+----------+------+-----------+-----------------+
|order_id|emp_id|order_date|amount|   category|prev_order_amount|
+--------+------+----------+------+-----------+-----------------+
|     101|     1|2024-01-05| 500.0|Electronics|              0.0|
|     104|     1|2024-02-20| 850.0|Electronics|            500.0|
|     110|     1|2024-05-02|1100.0|Electronics|            850.0|
|     103|     2|2024-02-10| 300.0|   Clothing|              0.0|
|     108|     2|2024-04-10| 700.0|Electronics|            300.0|
|     102|     3|2024-01-18|1200.0|Electronics|              0.0|
|     106|     3|2024-03-15| 450.0|   Clothing|           1200.0|
|     111|     4|2024-05-15| 200.0|   Clothing|              0.0|
|     105|     5|2024-03-01|2000.0|  Furniture|              0.0|
|     112|     6|2024-06-01| 600.0|Electronics|              0.0|
|     107|     7|2024-03-22| 150.0|   Clothing|              0.0|
|     109|     9|2024-04-25| 950.0|  Furniture|              0.0|
+--------+

# Part 6 — String & Date Functions

In [72]:
# 6.1 Add a column name_upper (uppercase of name) and dept_short (first 3 characters of department, uppercased).

employees=employees.withColumn("name_upper",upper(col("employee_name")))\
    .withColumn("dept_short",upper(substring(col("department"),1,3)))

employees.show()

+------+-------------+-----------+-------+------+----------+---------+----------------+---------+-----------+----------+----------+
|emp_id|employee_name| department|country|salary| join_date|is_active|years_of_service|is_senior|salary_band|name_upper|dept_short|
+------+-------------+-----------+-------+------+----------+---------+----------------+---------+-----------+----------+----------+
|     1|        Alice|Engineering|  India| 95000|2019-03-15|     true|               7|     true|        Mid|     ALICE|       ENG|
|     2|          Bob|  Marketing|    USA| 72000|2020-07-01|     true|               5|     true|        Mid|       BOB|       MAR|
|     3|        Carol|Engineering|    USA|110000|2018-11-20|     true|               7|     true|       High|     CAROL|       ENG|
|     4|        David|         HR|  India| 60000|2021-01-10|    false|               5|     true|        Low|     DAVID|        HR|
|     5|          Eva|Engineering|     UK| 98000|2017-06-05|     true|      

In [73]:
# 6.2 Extract join_year, join_month from join_date as separate integer columns.

employees.withColumn("join_year", year(col("join_date"))) \
       .withColumn("join_month", month(col("join_date"))).show()


+------+-------------+-----------+-------+------+----------+---------+----------------+---------+-----------+----------+----------+---------+----------+
|emp_id|employee_name| department|country|salary| join_date|is_active|years_of_service|is_senior|salary_band|name_upper|dept_short|join_year|join_month|
+------+-------------+-----------+-------+------+----------+---------+----------------+---------+-----------+----------+----------+---------+----------+
|     1|        Alice|Engineering|  India| 95000|2019-03-15|     true|               7|     true|        Mid|     ALICE|       ENG|     2019|         3|
|     2|          Bob|  Marketing|    USA| 72000|2020-07-01|     true|               5|     true|        Mid|       BOB|       MAR|     2020|         7|
|     3|        Carol|Engineering|    USA|110000|2018-11-20|     true|               7|     true|       High|     CAROL|       ENG|     2018|        11|
|     4|        David|         HR|  India| 60000|2021-01-10|    false|            

In [74]:
# 6.3 From orders, extract the order_month and find the month with the highest total revenue.

orders.withColumn("order_month",month(col("order_date")))\
    .groupBy("order_month")\
    .agg(sum("amount").alias("total_amount"))\
    .sort(col("total_amount").desc())\
    .show(1)

+-----------+------------+
|order_month|total_amount|
+-----------+------------+
|          3|      2600.0|
+-----------+------------+
only showing top 1 row


In [75]:
# 6.4 Add a column name_length (character count of name) and filter to employees whose name is longer than 4 characters.

employees.withColumn("name_length", length(col("employee_name")))\
            .filter(col("name_length") > 4).show()



+------+-------------+-----------+-------+------+----------+---------+----------------+---------+-----------+----------+----------+-----------+
|emp_id|employee_name| department|country|salary| join_date|is_active|years_of_service|is_senior|salary_band|name_upper|dept_short|name_length|
+------+-------------+-----------+-------+------+----------+---------+----------------+---------+-----------+----------+----------+-----------+
|     1|        Alice|Engineering|  India| 95000|2019-03-15|     true|               7|     true|        Mid|     ALICE|       ENG|          5|
|     3|        Carol|Engineering|    USA|110000|2018-11-20|     true|               7|     true|       High|     CAROL|       ENG|          5|
|     4|        David|         HR|  India| 60000|2021-01-10|    false|               5|     true|        Low|     DAVID|        HR|          5|
|     6|        Frank|  Marketing|  India| 68000|2022-03-22|     true|               4|    false|        Mid|     FRANK|       MAR|     

# Part 7 — SQL Queries


In [76]:
employees.createOrReplaceTempView("employees")
orders.createOrReplaceTempView("orders")

In [77]:
# 7.1 Write a SQL query to get the top 3 highest-paid employees per department.

spark.sql("""
    with cte as(select *,DENSE_RANK() over (partition by department order by salary) as rnk from employees)
    select * from cte where rnk in (1,2,3)
""").show()

+------+-------------+-----------+-------+------+----------+---------+----------------+---------+-----------+----------+----------+---+
|emp_id|employee_name| department|country|salary| join_date|is_active|years_of_service|is_senior|salary_band|name_upper|dept_short|rnk|
+------+-------------+-----------+-------+------+----------+---------+----------------+---------+-----------+----------+----------+---+
|     1|        Alice|Engineering|  India| 95000|2019-03-15|     true|               7|     true|        Mid|     ALICE|       ENG|  1|
|     5|          Eva|Engineering|     UK| 98000|2017-06-05|     true|               8|     true|        Mid|       EVA|       ENG|  2|
|     3|        Carol|Engineering|    USA|110000|2018-11-20|     true|               7|     true|       High|     CAROL|       ENG|  3|
|    10|         Jake|         HR|    USA| 58000|2023-02-18|     true|               3|    false|        Low|      JAKE|        HR|  1|
|     4|        David|         HR|  India| 60000

In [78]:
# 7.2 Write a SQL query to find all employees from India or UK who joined before 2021.

spark.sql("""
    select * from employees where country in ('India','UK') and year(join_date)<2021
""").show()

+------+-------------+-----------+-------+------+----------+---------+----------------+---------+-----------+----------+----------+
|emp_id|employee_name| department|country|salary| join_date|is_active|years_of_service|is_senior|salary_band|name_upper|dept_short|
+------+-------------+-----------+-------+------+----------+---------+----------------+---------+-----------+----------+----------+
|     1|        Alice|Engineering|  India| 95000|2019-03-15|     true|               7|     true|        Mid|     ALICE|       ENG|
|     5|          Eva|Engineering|     UK| 98000|2017-06-05|     true|               8|     true|        Mid|       EVA|       ENG|
|     8|         Hank|Engineering|     UK|120000|2016-12-01|    false|               9|     true|       High|      HANK|       ENG|
+------+-------------+-----------+-------+------+----------+---------+----------------+---------+-----------+----------+----------+



In [79]:
# 7.3 Write a SQL query that joins employees and orders, groups by department and category, and returns total revenue per combination. 
#   Sort by department, then total_revenue descending.


spark.sql("""
    select department,category,sum(amount) as total_revenue 
    from employees e join orders o on e.emp_id=o.emp_id
    group by department, category
    order by department, total_revenue desc
""").show()

[Stage 205:==========================================>              (6 + 2) / 8]

+-----------+-----------+-------------+
| department|   category|total_revenue|
+-----------+-----------+-------------+
|Engineering|Electronics|       3650.0|
|Engineering|  Furniture|       2000.0|
|Engineering|   Clothing|        450.0|
|         HR|   Clothing|        350.0|
|  Marketing|Electronics|       1300.0|
|  Marketing|  Furniture|        950.0|
|  Marketing|   Clothing|        300.0|
+-----------+-----------+-------------+



# Part 8 — Handling Nulls & Duplicates

In [80]:
dirty_data = [
    (11, "Leo",   None,          "USA",  None,  "2020-01-01", True),
    (12, "Mia",   "Engineering", None,   88000, None,         True),
    (1,  "Alice", "Engineering", "India",95000, "2019-03-15", True),  # duplicate
]

employees_schema_2 = StructType([
    StructField("emp_id",     IntegerType(), False),
    StructField("name",       StringType(),  False),
    StructField("department", StringType(),  True),
    StructField("country",    StringType(),  True),
    StructField("salary",     IntegerType(), True),
    StructField("join_date",  StringType(),  True),
    StructField("is_active",  BooleanType(), False),
])

dirty_df = spark.createDataFrame(dirty_data, employees_schema_2)

employees = spark.createDataFrame(employees_data, employees_schema)


employees_dirty = employees.union(dirty_df)

employees_dirty.show(truncate=False)

+------+-----+-----------+-------+------+----------+---------+
|emp_id|name |department |country|salary|join_date |is_active|
+------+-----+-----------+-------+------+----------+---------+
|1     |Alice|Engineering|India  |95000 |2019-03-15|true     |
|2     |Bob  |Marketing  |USA    |72000 |2020-07-01|true     |
|3     |Carol|Engineering|USA    |110000|2018-11-20|true     |
|4     |David|HR         |India  |60000 |2021-01-10|false    |
|5     |Eva  |Engineering|UK     |98000 |2017-06-05|true     |
|6     |Frank|Marketing  |India  |68000 |2022-03-22|true     |
|7     |Grace|HR         |USA    |63000 |2020-09-14|true     |
|8     |Hank |Engineering|UK     |120000|2016-12-01|false    |
|9     |Ivy  |Marketing  |UK     |75000 |2021-05-30|true     |
|10    |Jake |HR         |USA    |58000 |2023-02-18|true     |
|11    |Leo  |NULL       |USA    |NULL  |2020-01-01|true     |
|12    |Mia  |Engineering|NULL   |88000 |NULL      |true     |
|1     |Alice|Engineering|India  |95000 |2019-03-15|tru

In [81]:
# 8.1 Count the total nulls in each column of employees_dirty.
employees_dirty.select([count(when(col(c).isNull(), c)).alias(c) for c in employees_dirty.columns]).show()






+------+----+----------+-------+------+---------+---------+
|emp_id|name|department|country|salary|join_date|is_active|
+------+----+----------+-------+------+---------+---------+
|     0|   0|         1|      1|     1|        1|        0|
+------+----+----------+-------+------+---------+---------+



In [82]:
# 8.2 Fill null department with "Unknown" and null salary with the median salary of the existing employees (compute it first, then use fillna).
median_salary = employees_dirty.approxQuantile("salary", [0.5], 0.01)[0]
employees_dirty.fillna({"department": "Unknown", "salary": median_salary}).show()


+------+-----+-----------+-------+------+----------+---------+
|emp_id| name| department|country|salary| join_date|is_active|
+------+-----+-----------+-------+------+----------+---------+
|     1|Alice|Engineering|  India| 95000|2019-03-15|     true|
|     2|  Bob|  Marketing|    USA| 72000|2020-07-01|     true|
|     3|Carol|Engineering|    USA|110000|2018-11-20|     true|
|     4|David|         HR|  India| 60000|2021-01-10|    false|
|     5|  Eva|Engineering|     UK| 98000|2017-06-05|     true|
|     6|Frank|  Marketing|  India| 68000|2022-03-22|     true|
|     7|Grace|         HR|    USA| 63000|2020-09-14|     true|
|     8| Hank|Engineering|     UK|120000|2016-12-01|    false|
|     9|  Ivy|  Marketing|     UK| 75000|2021-05-30|     true|
|    10| Jake|         HR|    USA| 58000|2023-02-18|     true|
|    11|  Leo|    Unknown|    USA| 75000|2020-01-01|     true|
|    12|  Mia|Engineering|   NULL| 88000|      NULL|     true|
|     1|Alice|Engineering|  India| 95000|2019-03-15|   

In [83]:
# 8.3 Drop rows where country is null.

employees_clean = employees_dirty.dropna(subset=["country"])
employees_clean.show()

+------+-----+-----------+-------+------+----------+---------+
|emp_id| name| department|country|salary| join_date|is_active|
+------+-----+-----------+-------+------+----------+---------+
|     1|Alice|Engineering|  India| 95000|2019-03-15|     true|
|     2|  Bob|  Marketing|    USA| 72000|2020-07-01|     true|
|     3|Carol|Engineering|    USA|110000|2018-11-20|     true|
|     4|David|         HR|  India| 60000|2021-01-10|    false|
|     5|  Eva|Engineering|     UK| 98000|2017-06-05|     true|
|     6|Frank|  Marketing|  India| 68000|2022-03-22|     true|
|     7|Grace|         HR|    USA| 63000|2020-09-14|     true|
|     8| Hank|Engineering|     UK|120000|2016-12-01|    false|
|     9|  Ivy|  Marketing|     UK| 75000|2021-05-30|     true|
|    10| Jake|         HR|    USA| 58000|2023-02-18|     true|
|    11|  Leo|       NULL|    USA|  NULL|2020-01-01|     true|
|     1|Alice|Engineering|  India| 95000|2019-03-15|     true|
+------+-----+-----------+-------+------+----------+---

In [84]:
# 8.4 Remove duplicate rows based on emp_id, keeping the first occurrence.

employees1 = employees_clean.dropDuplicates(["emp_id"])
employees1.show()

+------+-----+-----------+-------+------+----------+---------+
|emp_id| name| department|country|salary| join_date|is_active|
+------+-----+-----------+-------+------+----------+---------+
|     1|Alice|Engineering|  India| 95000|2019-03-15|     true|
|     2|  Bob|  Marketing|    USA| 72000|2020-07-01|     true|
|     3|Carol|Engineering|    USA|110000|2018-11-20|     true|
|     4|David|         HR|  India| 60000|2021-01-10|    false|
|     5|  Eva|Engineering|     UK| 98000|2017-06-05|     true|
|     6|Frank|  Marketing|  India| 68000|2022-03-22|     true|
|     7|Grace|         HR|    USA| 63000|2020-09-14|     true|
|     8| Hank|Engineering|     UK|120000|2016-12-01|    false|
|     9|  Ivy|  Marketing|     UK| 75000|2021-05-30|     true|
|    10| Jake|         HR|    USA| 58000|2023-02-18|     true|
|    11|  Leo|       NULL|    USA|  NULL|2020-01-01|     true|
+------+-----+-----------+-------+------+----------+---------+



# Part 9 — Persisting & Partitioning (local filesystem)


In [85]:
# 9.1 Write the joined DataFrame from Task 4.1 to /tmp/pyspark_assignment/orders_enriched/ as Parquet, partitioned by department.


emp_df.write.partitionBy("department").mode("overwrite").parquet("/tmp/pyspark_assignment/orders_enriched/")

In [87]:
# 9.2 Read the Parquet files back and verify the row count matches the original.

df = spark.read.parquet("/tmp/pyspark_assignment/orders_enriched/")

df.show(truncate=False)

# Compare row counts
original_count = emp_df.count()
read_count = df.count()

print(f"Original row count: {original_count}")
print(f"Read row count    : {read_count}")


+--------+-------------+------+-----------+----------+-----------+
|order_id|employee_name|amount|category   |order_date|department |
+--------+-------------+------+-----------+----------+-----------+
|101     |Alice        |500.0 |Electronics|2024-01-05|Engineering|
|104     |Alice        |850.0 |Electronics|2024-02-20|Engineering|
|110     |Alice        |1100.0|Electronics|2024-05-02|Engineering|
|102     |Carol        |1200.0|Electronics|2024-01-18|Engineering|
|106     |Carol        |450.0 |Clothing   |2024-03-15|Engineering|
|105     |Eva          |2000.0|Furniture  |2024-03-01|Engineering|
|103     |Bob          |300.0 |Clothing   |2024-02-10|Marketing  |
|108     |Bob          |700.0 |Electronics|2024-04-10|Marketing  |
|112     |Frank        |600.0 |Electronics|2024-06-01|Marketing  |
|109     |Ivy          |950.0 |Furniture  |2024-04-25|Marketing  |
|111     |David        |200.0 |Clothing   |2024-05-15|HR         |
|107     |Grace        |150.0 |Clothing   |2024-03-22|HR      

[Stage 255:=====================>                                   (3 + 5) / 8]

Original row count: 12
Read row count    : 12


In [88]:
# 9.3 Write the employees DataFrame as a single CSV file (use coalesce(1)) with a header to /tmp/pyspark_assignment/employees.csv.


employees.coalesce(1).write.mode('overwrite').option('header' , 'true').csv("/tmp/pyspark_assignment/employees.csv")



In [90]:
# 9.4 Read the CSV back with inferSchema=True and print its schema.

df4 = spark.read.option('header' , 'true')\
                .option("inferSchema",True) \
                .option("delimiter",",") \
  .csv("/tmp/pyspark_assignment/employees.csv")

df4.printSchema()


root
 |-- emp_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- country: string (nullable = true)
 |-- salary: integer (nullable = true)
 |-- join_date: date (nullable = true)
 |-- is_active: boolean (nullable = true)



# Part 10 — UDFs

In [92]:

# 10.1 Write a Python UDF categorize_salary(salary) that returns:

# "Entry" if salary < 65,000
# "Mid" if 65,000 ≤ salary < 100,000
# "Senior" if salary ≥ 100,000
# Register it and apply it to the employees DataFrame.



@udf(returnType=StringType()) 
def categorize_salary(sal):
    if sal<65000:
        return "Entry"
    elif sal>=65000 and sal<100000:
        return "Mid"
    else:
        return "Senior"

employees=employees.withColumn("Salary_category", categorize_salary(col("salary")))

employees.show()



    




+------+-----+-----------+-------+------+----------+---------+---------------+
|emp_id| name| department|country|salary| join_date|is_active|Salary_category|
+------+-----+-----------+-------+------+----------+---------+---------------+
|     1|Alice|Engineering|  India| 95000|2019-03-15|     true|            Mid|
|     2|  Bob|  Marketing|    USA| 72000|2020-07-01|     true|            Mid|
|     3|Carol|Engineering|    USA|110000|2018-11-20|     true|         Senior|
|     4|David|         HR|  India| 60000|2021-01-10|    false|          Entry|
|     5|  Eva|Engineering|     UK| 98000|2017-06-05|     true|            Mid|
|     6|Frank|  Marketing|  India| 68000|2022-03-22|     true|            Mid|
|     7|Grace|         HR|    USA| 63000|2020-09-14|     true|          Entry|
|     8| Hank|Engineering|     UK|120000|2016-12-01|    false|         Senior|
|     9|  Ivy|  Marketing|     UK| 75000|2021-05-30|     true|            Mid|
|    10| Jake|         HR|    USA| 58000|2023-02-18|

In [93]:
# 10.2 Write a UDF mask_name(name) that returns the first character followed by asterisks for the rest (e.g. "Alice" → "A****"). Apply it to employees.

@udf(returnType=StringType()) 
def mask_name(name):
    return name[0] + '*' * (len(name)-1)


employees=employees.withColumn("Masked_name", mask_name(col("name")))

employees.show()


+------+-----+-----------+-------+------+----------+---------+---------------+-----------+
|emp_id| name| department|country|salary| join_date|is_active|Salary_category|Masked_name|
+------+-----+-----------+-------+------+----------+---------+---------------+-----------+
|     1|Alice|Engineering|  India| 95000|2019-03-15|     true|            Mid|      A****|
|     2|  Bob|  Marketing|    USA| 72000|2020-07-01|     true|            Mid|        B**|
|     3|Carol|Engineering|    USA|110000|2018-11-20|     true|         Senior|      C****|
|     4|David|         HR|  India| 60000|2021-01-10|    false|          Entry|      D****|
|     5|  Eva|Engineering|     UK| 98000|2017-06-05|     true|            Mid|        E**|
|     6|Frank|  Marketing|  India| 68000|2022-03-22|     true|            Mid|      F****|
|     7|Grace|         HR|    USA| 63000|2020-09-14|     true|          Entry|      G****|
|     8| Hank|Engineering|     UK|120000|2016-12-01|    false|         Senior|       H***|